<a href="https://colab.research.google.com/github/LayonFornaciari/puc-mvp-credito/blob/main/puc_mvp_credito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP: Previsão de Risco de Crédito (Inadimplência)
**Sprint:** Qualidade de Software, Segurança e Sistemas Inteligentes
**Autor:** Layon Fornaciari

## 1. Definição do Problema e Reflexão sobre Segurança (LGPD)
O objetivo deste modelo de Machine Learning é prever se um cliente bancário tem alto risco de inadimplência (classificado como "bad" risk) com base em suas informações financeiras e pessoais.
A identificação precoce de maus pagadores reduz perdas financeiras (falsos negativos) e otimiza a aprovação de bons clientes.

**Reflexão sobre Segurança da Informação e LGPD:**
No contexto real de uma instituição financeira, o dataset original contaria com PIIs (Personally Identifiable Information) como CPF, Nome Completo e Endereço. Para garantir a **Confidencialidade** (pilar da Tríade CID) e aderência à LGPD, este dataset foi previamente **anonimizado**. Variáveis de identificação direta foram removidas e variáveis sensíveis (como idade e sexo) são utilizadas unicamente para fins estatísticos e de aprovação de crédito, protegidas em ambiente restrito.

In [17]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 1. Carga dos Dados via URL do GitHub (Requisito da PUC)
url = 'https://raw.githubusercontent.com/LayonFornaciari/puc-mvp-credito/main/credit.csv'
dataset = pd.read_csv(url)

# Visualizando as primeiras linhas
display(dataset.head())

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_length,installment_rate,personal_status,other_debtors,...,property,age,installment_plan,housing,existing_credits,default,dependents,telephone,foreign_worker,job
0,< 0 DM,6,critical,radio/tv,1169,unknown,> 7 yrs,4,single male,none,...,real estate,67,none,own,2,1,1,yes,yes,skilled employee
1,1 - 200 DM,48,repaid,radio/tv,5951,< 100 DM,1 - 4 yrs,2,female,none,...,real estate,22,none,own,1,2,1,none,yes,skilled employee
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 yrs,2,single male,none,...,real estate,49,none,own,1,1,2,none,yes,unskilled resident
3,< 0 DM,42,repaid,furniture,7882,< 100 DM,4 - 7 yrs,2,single male,guarantor,...,building society savings,45,none,for free,1,1,2,none,yes,skilled employee
4,< 0 DM,24,delayed,car (new),4870,< 100 DM,1 - 4 yrs,3,single male,none,...,unknown/none,53,none,for free,2,2,2,none,yes,skilled employee


## 2. Pré-Processamento de Dados
Nesta etapa, preparamos os dados para os algoritmos:
1. Remoção de colunas inúteis (ID).
2. Tratamento de valores nulos (NaN).
3. Transformação da variável alvo (Risk) em binária: `Good = 0` e `Bad = 1` (queremos prever o risco alto).
4. Transformação de variáveis categóricas em numéricas (One-Hot Encoding).

In [18]:
# SEGURANÇA: Verificar se o dataset está vazio (caso a célula 1 não tenha sido recarregada)
if dataset.shape[0] == 0:
    raise ValueError("O dataset está vazio! Por favor, suba e RODE A CÉLULA DE CÓDIGO 1 novamente para recarregar os dados do zero.")

# Ajustando nomes das colunas para evitar erros de espaços em branco
dataset.columns = dataset.columns.str.strip()

# 1. Identificando a coluna alvo dinamicamente
target_col = None
for col in ['Risk', 'risk', 'Class', 'class', 'default', 'Creditability', 'Target']:
    if col in dataset.columns:
        target_col = col
        break

if target_col is None:
    target_col = dataset.columns[-1]
    print("⚠️ ATENÇÃO: A coluna de rótulos ('Risk') não foi encontrada no seu CSV!")
    print("Isso é comum em algumas versões deste dataset no Kaggle.")
    print(f"Estamos usando a última coluna ('{target_col}') como alvo temporário para não travar o MVP.\n")

print(f"Coluna alvo identificada para treinamento: '{target_col}'")

# 2. Removendo a coluna de índice que costuma vir no dataset do Kaggle
if 'Unnamed: 0' in dataset.columns:
    dataset = dataset.drop('Unnamed: 0', axis=1)

# 3. Tratando valores nulos (se as colunas existirem no dataset)
if 'Saving accounts' in dataset.columns:
    dataset['Saving accounts'] = dataset['Saving accounts'].fillna('No Account')
if 'Checking account' in dataset.columns:
    dataset['Checking account'] = dataset['Checking account'].fillna('No Account')

# 4. Mapeando a variável alvo com segurança
if dataset[target_col].dtype == 'object':
    # Se for texto, limpa e padroniza
    dataset[target_col] = dataset[target_col].astype(str).str.strip().str.lower()

    # Se os valores forem exatamente good/bad, mapeia
    if set(dataset[target_col].unique()).issubset({'good', 'bad'}):
        dataset[target_col] = dataset[target_col].map({'good': 0, 'bad': 1})
    else:
        # Fallback seguro: converte qualquer outra classe para números sequenciais
        dataset[target_col] = dataset[target_col].astype('category').cat.codes
else:
    # Se já for numérico, verifica os valores únicos antes de tocar neles
    valores = set(dataset[target_col].dropna().unique())
    if valores.issubset({1, 2}):
        dataset[target_col] = dataset[target_col].map({1: 0, 2: 1})
    elif not valores.issubset({0, 1}):
        # Se for numérico, mas não for (0, 1) nem (1, 2), usamos o codificador automático
        dataset[target_col] = dataset[target_col].astype('category').cat.codes

# Tratamento de segurança final: remove apenas se ficou NaN na coluna alvo
dataset = dataset.dropna(subset=[target_col])

# 5. Transformando variáveis categóricas em numéricas (One-Hot Encoding)
dataset_encoded = pd.get_dummies(dataset, drop_first=True)

# Separando X (features) e y (target)
X = dataset_encoded.drop(target_col, axis=1)
y = dataset_encoded[target_col]

print(f"Formato final de X: {X.shape}")
print(f"Distribuição do Target:\n{y.value_counts()}")

Coluna alvo identificada para treinamento: 'default'
Formato final de X: (1000, 48)
Distribuição do Target:
default
0    700
1    300
Name: count, dtype: int64


## 3. Separação em Treino e Teste (Holdout)
Separamos 80% dos dados para treinar o modelo e 20% para testá-lo em dados nunca vistos.

In [19]:
from sklearn.model_selection import train_test_split

# Separação Holdout (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Dados separados com sucesso!")

Dados separados com sucesso!


## 4. Modelagem com Pipelines
Vamos treinar os 4 modelos exigidos no MVP utilizando `Pipeline` do Scikit-Learn.
O Pipeline garante que a padronização (`StandardScaler`) seja aplicada corretamente tanto no treino quanto no teste, evitando vazamento de dados (data leakage).

**Modelos a serem avaliados:**
1. KNN (K-Nearest Neighbors)
2. Decision Tree (Árvore de Decisão)
3. Naive Bayes (GaussianNB)
4. SVM (Support Vector Machine)

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Criando os pipelines
pipelines = {
    'KNN': Pipeline([('scaler', StandardScaler()), ('classifier', KNeighborsClassifier())]),
    'Decision Tree': Pipeline([('scaler', StandardScaler()), ('classifier', DecisionTreeClassifier(random_state=42))]),
    'Naive Bayes': Pipeline([('scaler', StandardScaler()), ('classifier', GaussianNB())]),
    'SVM': Pipeline([('scaler', StandardScaler()), ('classifier', SVC(random_state=42))])
}

# Treinando e avaliando cada modelo
resultados = {}

for nome, modelo in pipelines.items():
    # Treinamento
    modelo.fit(X_train, y_train)

    # Predição
    y_pred = modelo.predict(X_test)

    # Avaliação
    print(f"\n--- Resultados para {nome} ---")
    print(classification_report(y_test, y_pred))


--- Resultados para KNN ---
              precision    recall  f1-score   support

           0       0.76      0.89      0.82       140
           1       0.57      0.35      0.43        60

    accuracy                           0.72       200
   macro avg       0.66      0.62      0.63       200
weighted avg       0.70      0.72      0.70       200


--- Resultados para Decision Tree ---
              precision    recall  f1-score   support

           0       0.75      0.70      0.73       140
           1       0.40      0.47      0.43        60

    accuracy                           0.63       200
   macro avg       0.58      0.58      0.58       200
weighted avg       0.65      0.63      0.64       200


--- Resultados para Naive Bayes ---
              precision    recall  f1-score   support

           0       0.83      0.71      0.76       140
           1       0.49      0.67      0.57        60

    accuracy                           0.69       200
   macro avg       0.66

## 5. Otimização de Hiperparâmetros (GridSearchCV)
Em problemas de risco de crédito, a acurácia global não é a melhor métrica. Estamos mais interessados no **Recall da classe 1 (Bad Risk)**. Um falso negativo (dizer que um cliente é bom pagador quando ele é mau pagador) gera grande prejuízo.

Vamos otimizar o modelo **SVM** utilizando `GridSearchCV` e Validação Cruzada (Cross-Validation) focando em melhorar o Recall.

In [21]:
from sklearn.model_selection import GridSearchCV

# Definindo o pipeline do SVM para otimização
pipe_svm = Pipeline([('scaler', StandardScaler()), ('classifier', SVC(probability=True, random_state=42))])

# Definindo a grade de hiperparâmetros (Grid)
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf'],
    'classifier__gamma': ['scale', 'auto']
}

# Executando o GridSearchCV (5 folds de validação cruzada)
grid = GridSearchCV(pipe_svm, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Melhores parâmetros encontrados: {grid.best_params_}")

# Avaliando o melhor modelo encontrado
melhor_modelo = grid.best_estimator_
y_pred_otimizado = melhor_modelo.predict(X_test)

print("\n--- Resultados SVM Otimizado ---")
print(classification_report(y_test, y_pred_otimizado))
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred_otimizado))

Melhores parâmetros encontrados: {'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}

--- Resultados SVM Otimizado ---
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       140
           1       0.62      0.62      0.62        60

    accuracy                           0.77       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.77      0.77      0.77       200

Matriz de Confusão:
[[117  23]
 [ 23  37]]


## 6. Exportação do Modelo e Conclusões Finais

**Análise dos Resultados:**
Observamos que o algoritmo otimizado obteve um balanço aceitável. Embora o recall da classe 1 (Inadimplentes) seja um desafio comum em bases desbalanceadas de crédito, a padronização via Pipeline e a otimização com GridSearchCV garantiram um modelo robusto e blindado contra data leakage.

Este modelo otimizado será agora exportado e embarcado na nossa API de Back-end utilizando a biblioteca `joblib`.

In [22]:
import joblib

# Exportando o melhor modelo (o pipeline completo que inclui o StandardScaler e o classificador)
arquivo_modelo = 'modelo_credito.pkl'
joblib.dump(melhor_modelo, arquivo_modelo)

print(f"Modelo exportado com sucesso para o arquivo: {arquivo_modelo}")
# O arquivo ficará salvo na aba "Arquivos" do lado esquerdo do Colab.
# Faça o download dele para o seu computador!

Modelo exportado com sucesso para o arquivo: modelo_credito.pkl


In [23]:
# Exportando as colunas originais do treinamento para a API
joblib.dump(list(X.columns), 'colunas.pkl')
print("Colunas exportadas com sucesso!")

Colunas exportadas com sucesso!
